In [1]:
#| default_exp core.attention
#| export

In [2]:
#| export

import numpy as np
import math
import time
from typing import Optional, Tuple, List

# Import dependencies from previous modules - following TinyTorch dependency chain
from tinytorch.core.tensor import Tensor
from tinytorch.core.layers import Linear
from tinytorch.core.activations import Softmax

# Constants for attention computation
MASK_VALUE = -1e9  # Large negative value used for attention masking (becomes ~0 after softmax)

In [3]:
#| export
def _compute_attention_scores(Q: Tensor, K: Tensor) -> Tensor:
    """Compute raw attention scores via Q @ K^T.

    TODO: Transpose K and multiply by Q to get similarity matrix

    APPROACH:
    1. Transpose K: swap last two dims so (batch, seq, d) -> (batch, d, seq)
    2. Matrix multiply: Q @ K^T gives (batch, seq, seq) scores

    EXAMPLE:
    >>> Q = Tensor(np.random.randn(1, 3, 4))  # 3 tokens, dim=4
    >>> K = Tensor(np.random.randn(1, 3, 4))
    >>> scores = _compute_attention_scores(Q, K)
    >>> print(scores.shape)  # (1, 3, 3) -- every token scored against every other

    HINT: Use K.transpose(-2, -1) to swap the last two dimensions
    """
    ### BEGIN SOLUTION
    K_t = K.transpose(-2, -1)
    return Q.matmul(K_t)
    ### END SOLUTION

In [4]:
def test_unit_attention_scores():
    """🧪 Test attention score computation."""
    print("🧪 Unit Test: Attention Scores...")
    Q = Tensor(np.ones((1, 3, 4)))
    print(Q)
    K = Tensor(np.ones((1, 3, 4)))
    print(K)
    scores = _compute_attention_scores(Q, K)
    print(scores)
    assert scores.shape == (1, 3, 3), f"Expected (1,3,3), got {scores.shape}"
    assert np.allclose(scores.data, 4.0), "All-ones Q@K^T should give d_model=4"
    print("✅ Attention scores: correct shapes and values!")

if __name__ == "__main__":
    test_unit_attention_scores()

🧪 Unit Test: Attention Scores...
Tensor([[[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]])
Tensor([[[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]])
Tensor([[[4. 4. 4.]
  [4. 4. 4.]
  [4. 4. 4.]]])
✅ Attention scores: correct shapes and values!


In [5]:
#| export
def _scale_scores(scores: Tensor, d_model: int) -> Tensor:
    """Scale attention scores by 1/sqrt(d_model).

    TODO: Divide scores by the square root of the model dimension

    APPROACH:
    1. Compute scale factor: 1.0 / math.sqrt(d_model)
    2. Multiply scores by scale factor

    EXAMPLE:
    >>> scores = Tensor(np.array([[[4.0, 8.0]]]))
    >>> scaled = _scale_scores(scores, d_model=4)
    >>> print(scaled.data)  # [[[ 2.0, 4.0]]] -- divided by sqrt(4)=2

    HINT: Use math.sqrt() for the square root
    """
    ### BEGIN SOLUTION
    scale_factor = 1.0 / math.sqrt(d_model)
    return scores * scale_factor
    ### END SOLUTION

In [6]:
def test_unit_scale_scores():
    """🧪 Test attention score scaling."""
    print("🧪 Unit Test: Score Scaling...")
    scores = Tensor(np.array([[[4.0, 8.0]]]))
    scaled = _scale_scores(scores, d_model=4)
    # print(scaled)
    assert np.allclose(scaled.data, [[[2.0, 4.0]]]), f"Expected /sqrt(4)=2, got {scaled.data}"
    print("✅ Score scaling works correctly!")

if __name__ == "__main__":
    test_unit_scale_scores()

🧪 Unit Test: Score Scaling...
✅ Score scaling works correctly!


In [7]:
#| export
def _apply_mask(scores: Tensor, mask: Tensor) -> Tensor:
    """Apply causal mask by setting masked positions to -infinity.

    TODO: Add large negative values to positions where mask is 0

    APPROACH:
    1. Compute additive mask: (1 - mask) * MASK_VALUE
    2. Add to scores (masked positions become -inf, unmasked unchanged)

    EXAMPLE:
    >>> scores = Tensor(np.ones((1, 3, 3)))
    >>> mask = Tensor(np.tril(np.ones((1, 3, 3))))  # lower triangle
    >>> masked = _apply_mask(scores, mask)
    >>> print(masked.data[0, 0, 1])  # -1e9 (future position masked)

    HINT: mask=0 means "block this position", mask=1 means "allow"
    """
    ### BEGIN SOLUTION
    adder = (1.0 - mask.data) * MASK_VALUE
    return scores + Tensor(adder)
    ### END SOLUTION

In [8]:
def test_unit_apply_mask():
    """🧪 Test causal mask application."""
    print("🧪 Unit Test: Causal Masking...")
    scores = Tensor(np.ones((1, 3, 3)))
    mask = Tensor(np.tril(np.ones((1, 3, 3))))
    # print(mask)
    masked = _apply_mask(scores, mask)
    # print(masked)
    # Future positions should be large negative
    assert masked.data[0, 0, 1] < -1e8, "Future position not masked"
    # Past positions should be unchanged
    assert np.allclose(masked.data[0, 0, 0], 1.0), "Past position was modified"
    print("✅ Causal masking works correctly!")

if __name__ == "__main__":
    test_unit_apply_mask()

🧪 Unit Test: Causal Masking...
✅ Causal masking works correctly!


In [9]:
#| export
def scaled_dot_product_attention(Q: Tensor, K: Tensor, V: Tensor, mask: Optional[Tensor] = None) -> Tuple[Tensor, Tensor]:
    """Complete scaled dot-product attention.

    TODO: Compose the helpers into the full attention operation

    APPROACH:
    1. Call _compute_attention_scores(Q, K) for raw similarity
    2. Call _scale_scores(scores, Q.shape[-1]) for numerical stability
    3. If mask provided, call _apply_mask(scores, mask)
    4. Apply Softmax to get probability weights
    5. Multiply weights @ V for attended values

    SUB-PROBLEMS (you already implemented these):
    - _compute_attention_scores: Q @ K^T similarity matrix
    - _scale_scores: divide by sqrt(d) for stable softmax
    - _apply_mask: block future positions with -inf

    Args:
        Q: Query tensor of shape (batch_size, seq_len, d_model)
        K: Key tensor of shape (batch_size, seq_len, d_model)
        V: Value tensor of shape (batch_size, seq_len, d_model)
        mask: Optional causal mask, True=allow, False=mask (batch_size, seq_len, seq_len)

    Returns:
        output: Attended values (batch_size, seq_len, d_model)
        attention_weights: Attention matrix (batch_size, seq_len, seq_len)

    EXAMPLE:
    >>> Q = Tensor(np.random.randn(2, 4, 64))
    >>> K = Tensor(np.random.randn(2, 4, 64))
    >>> V = Tensor(np.random.randn(2, 4, 64))
    >>> output, weights = scaled_dot_product_attention(Q, K, V)
    >>> print(output.shape)   # (2, 4, 64)
    >>> print(weights.shape)  # (2, 4, 4)

    HINT: Softmax is already imported -- use Softmax()(scores, dim=-1)
    """
    ### BEGIN SOLUTION
    scores = _compute_attention_scores(Q, K)
    scores = _scale_scores(scores, Q.shape[-1])
    if mask is not None:
        scores = _apply_mask(scores, mask)
    softmax = Softmax()
    attention_weights = softmax(scores, dim=-1)
    output = attention_weights.matmul(V)
    return output, attention_weights
    ### END SOLUTION

In [10]:
def test_unit_scaled_dot_product_attention():
    """🧪 Test scaled dot-product attention implementation."""
    print("🧪 Unit Test: Scaled Dot-Product Attention...")

    # Test basic functionality
    batch_size, seq_len, d_model = 2, 4, 8
    Q = Tensor(np.random.randn(batch_size, seq_len, d_model))
    K = Tensor(np.random.randn(batch_size, seq_len, d_model))
    V = Tensor(np.random.randn(batch_size, seq_len, d_model))

    output, weights = scaled_dot_product_attention(Q, K, V)

    # Check output shapes
    assert output.shape == (batch_size, seq_len, d_model), f"Output shape {output.shape} incorrect"
    assert weights.shape == (batch_size, seq_len, seq_len), f"Weights shape {weights.shape} incorrect"

    # Check attention weights sum to 1 (probability distribution)
    weights_sum = weights.data.sum(axis=2)  # Sum over last dimension
    expected_sum = np.ones((batch_size, seq_len))
    assert np.allclose(weights_sum, expected_sum, atol=1e-6), "Attention weights don't sum to 1"

    # Test with causal mask
    mask = Tensor(np.tril(np.ones((batch_size, seq_len, seq_len)), k=0))  # Lower triangular
    output_masked, weights_masked = scaled_dot_product_attention(Q, K, V, mask)

    # Check that future positions have zero attention
    for b in range(batch_size):
        for i in range(seq_len):
            for j in range(i + 1, seq_len):  # Future positions
                assert abs(weights_masked.data[b, i, j]) < 1e-6, f"Future attention not masked at ({i},{j})"

    print("✅ scaled_dot_product_attention works correctly!")

if __name__ == "__main__":
    test_unit_scaled_dot_product_attention()

🧪 Unit Test: Scaled Dot-Product Attention...
✅ scaled_dot_product_attention works correctly!


In [11]:
#| export
class MultiHeadAttention:
    """
    Multi-head attention mechanism.

    Runs multiple attention heads in parallel, each learning different relationships.
    This is the core component of transformer architectures.
    """

    def __init__(self, embed_dim: int, num_heads: int):
        """
        Initialize multi-head attention.

        TODO: Set up linear projections and validate configuration

        APPROACH:
        1. Validate that embed_dim is divisible by num_heads
        2. Calculate head_dim (embed_dim // num_heads)
        3. Create linear layers for Q, K, V projections
        4. Create output projection layer
        5. Store configuration parameters

        Args:
            embed_dim: Embedding dimension (d_model)
            num_heads: Number of parallel attention heads

        EXAMPLE:
        >>> mha = MultiHeadAttention(embed_dim=512, num_heads=8)
        >>> mha.head_dim  # 64 (512 / 8)
        >>> len(mha.parameters())  # 4 linear layers * 2 params each = 8 tensors

        HINTS:
        - head_dim = embed_dim // num_heads must be integer
        - Need 4 Linear layers: q_proj, k_proj, v_proj, out_proj
        - Each projection maps embed_dim → embed_dim
        """
        ### BEGIN SOLUTION
        if embed_dim % num_heads != 0:
            raise ValueError(
                f"Multi-head attention dimension mismatch\n"
                f"  ❌ embed_dim={embed_dim} is not divisible by num_heads={num_heads} (remainder={embed_dim % num_heads})\n"
                f"  💡 Multi-head attention splits embed_dim equally among heads, so embed_dim must be a multiple of num_heads\n"
                f"  🔧 Try: embed_dim={num_heads * (embed_dim // num_heads + 1)} (next valid size) or num_heads={embed_dim // (embed_dim // num_heads)} (fewer heads)"
            )

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # linear projections for queries, keys, values
        self.q_proj = Linear(embed_dim, embed_dim)
        self.k_proj = Linear(embed_dim, embed_dim)
        self.v_proj = Linear(embed_dim, embed_dim)

        # output projection to mix information across heads
        self.out_proj = Linear(embed_dim, embed_dim)
        ### END SOLUTION

    def _split_heads(self, x: Tensor, batch_size: int, seq_len: int) -> Tensor:
        """Reshape to separate attention heads for parallel processing.

        TODO: Reshape (batch, seq, embed_dim) to (batch, heads, seq, head_dim)

        APPROACH:
        1. Reshape: (batch, seq, embed) -> (batch, seq, num_heads, head_dim)
        2. Transpose: swap seq and heads dims -> (batch, heads, seq, head_dim)

        EXAMPLE:
        >>> mha = MultiHeadAttention(embed_dim=64, num_heads=8)
        >>> x = Tensor(np.random.randn(2, 10, 64))  # batch=2, seq=10
        >>> split = mha._split_heads(x, 2, 10)
        >>> print(split.shape)  # (2, 8, 10, 8) -- 8 heads of dim 8

        HINT: reshape(batch, seq, heads, head_dim) then transpose(1, 2)
        """
        ### BEGIN SOLUTION
        x = x.reshape(batch_size, seq_len, self.num_heads, self.head_dim)
        return x.transpose(1, 2)
        ### END SOLUTION

    def _merge_heads(self, x: Tensor, batch_size: int, seq_len: int) -> Tensor:
        """Merge attention heads back into single embedding dimension.

        TODO: Reshape (batch, heads, seq, head_dim) to (batch, seq, embed_dim)

        APPROACH:
        1. Transpose: swap heads and seq -> (batch, seq, heads, head_dim)
        2. Reshape: merge last two dims -> (batch, seq, embed_dim)

        EXAMPLE:
        >>> # After attention with 8 heads of dim 8:
        >>> attended = Tensor(np.random.randn(2, 8, 10, 8))
        >>> merged = mha._merge_heads(attended, 2, 10)
        >>> print(merged.shape)  # (2, 10, 64) -- back to embed_dim

        HINT: transpose(1, 2) then reshape(batch, seq, embed_dim)
        """
        ### BEGIN SOLUTION
        x = x.transpose(1, 2)
        return x.reshape(batch_size, seq_len, self.embed_dim)
        ### END SOLUTION

    def forward(self, x: Tensor, mask: Optional[Tensor] = None) -> Tensor:
        """
        Forward pass through multi-head attention.

        TODO: Compose the helpers into the complete multi-head attention forward pass

        APPROACH:
        1. Extract input dimensions and validate embed_dim
        2. Project input to Q, K, V using linear layers
        3. Call _split_heads() to separate into parallel heads
        4. Apply scaled_dot_product_attention to all heads at once
        5. Call _merge_heads() to recombine heads
        6. Apply output projection

        SUB-PROBLEMS (you already implemented these):
        - _split_heads: reshape 3D -> 4D for parallel head processing
        - _merge_heads: reshape 4D -> 3D to recombine head outputs

        Args:
            x: Input tensor (batch_size, seq_len, embed_dim)
            mask: Optional attention mask (batch_size, seq_len, seq_len)

        Returns:
            output: Attended representation (batch_size, seq_len, embed_dim)

        EXAMPLE:
        >>> mha = MultiHeadAttention(embed_dim=64, num_heads=8)
        >>> x = Tensor(np.random.randn(2, 10, 64))  # batch=2, seq=10, dim=64
        >>> output = mha.forward(x)
        >>> print(output.shape)  # (2, 10, 64) - same as input

        HINT: Use scaled_dot_product_attention for the attention computation
        """
        ### BEGIN SOLUTION
        # Step 1: Extract dimensions and validate
        batch_size, seq_len, embed_dim = x.shape
        if embed_dim != self.embed_dim:
            raise ValueError(
                f"MultiHeadAttention input dimension mismatch\n"
                f"  ❌ Expected embed_dim={self.embed_dim}, got {embed_dim} from input shape {x.shape}\n"
                f"  💡 The last dimension of input must match embed_dim from initialization (MultiHeadAttention({self.embed_dim}, {self.num_heads}))\n"
                f"  🔧 Try: x.reshape({x.shape[0]}, {x.shape[1]}, {self.embed_dim}) or create new MultiHeadAttention({embed_dim}, num_heads)"
            )

        # step 2: project to Q, K, V
        Q = self.q_proj.forward(x)
        K = self.k_proj.forward(x)
        V = self.v_proj.forward(x)

        # step 3: split into heads
        Q = self._split_heads(Q, batch_size, seq_len)
        K = self._split_heads(K, batch_size, seq_len)
        V = self._split_heads(V, batch_size, seq_len)

        # step 4: apply attention (reshape mask for head broadcasting)
        mask_reshaped = mask
        if mask is not None and len(mask.shape) == 3:
            batch_size_mask, seq_len_mask, _ = mask.shape
            mask_data = mask.data.reshape(batch_size_mask, 1, seq_len_mask, seq_len_mask)
            mask_reshaped = Tensor(mask_data)

        attended, _ = scaled_dot_product_attention(Q, K, V, mask=mask_reshaped)

        # step 5: merge heads back together
        concat_output = self._merge_heads(attended, batch_size, seq_len)

        # step 6: apply output projection
        output = self.out_proj.forward(concat_output)

        return output
        ### END SOLUTION

    def __call__(self, x: Tensor, mask: Optional[Tensor] = None) -> Tensor:
        """Make MultiHeadAttention callable like attention(x)."""
        return self.forward(x, mask)

    def parameters(self) -> List[Tensor]:
        """
        Return all trainable parameters.

        TODO: Collect parameters from all linear layers

        APPROACH:
        1. Get parameters from q_proj, k_proj, v_proj, out_proj
        2. Combine into single list

        Returns:
            List of all parameter tensors

        EXAMPLE:
        >>> mha = MultiHeadAttention(embed_dim=64, num_heads=8)
        >>> params = mha.parameters()
        >>> print(len(params))  # 8 (4 layers × 2 params each: weight + bias)
        >>> print(params[0].shape)  # (64, 64) - q_proj weight
        >>> print(params[1].shape)  # (64,) - q_proj bias

        HINTS:
        - Each Linear layer has .parameters() method that returns [weight, bias]
        - Use extend() to add all parameters from each layer to the list
        - Total should be 8 tensors: 4 layers × 2 parameters each
        """
        ### BEGIN SOLUTION
        params = []
        params.extend(self.q_proj.parameters())
        params.extend(self.k_proj.parameters())
        params.extend(self.v_proj.parameters())
        params.extend(self.out_proj.parameters())
        return params
        ### END SOLUTION

In [12]:
def test_unit_split_heads():
    """🧪 Test head splitting reshape."""
    print("🧪 Unit Test: Split Heads...")
    mha = MultiHeadAttention(embed_dim=64, num_heads=8)
    x = Tensor(np.random.randn(2, 10, 64))
    split = mha._split_heads(x, 2, 10)
    assert split.shape == (2, 8, 10, 8), f"Expected (2,8,10,8), got {split.shape}"
    print("✅ Split heads: correct 4D shape!")

if __name__ == "__main__":
    test_unit_split_heads()

🧪 Unit Test: Split Heads...
✅ Split heads: correct 4D shape!


In [13]:
def test_unit_merge_heads():
    """🧪 Test head merging reshape."""
    print("🧪 Unit Test: Merge Heads...")
    mha = MultiHeadAttention(embed_dim=64, num_heads=8)
    # Create 4D tensor as if from split_heads
    x_4d = Tensor(np.random.randn(2, 8, 10, 8))
    merged = mha._merge_heads(x_4d, 2, 10)
    assert merged.shape == (2, 10, 64), f"Expected (2,10,64), got {merged.shape}"

    # Verify round-trip: split then merge recovers original data
    original = Tensor(np.random.randn(2, 10, 64))
    split = mha._split_heads(original, 2, 10)
    recovered = mha._merge_heads(split, 2, 10)
    assert np.allclose(original.data, recovered.data), "Split->merge should recover original data"
    print("✅ Merge heads: correct 3D shape and round-trip!")

if __name__ == "__main__":
    test_unit_merge_heads()

🧪 Unit Test: Merge Heads...
✅ Merge heads: correct 3D shape and round-trip!


In [14]:
def test_unit_multihead_attention():
    """🧪 Test multi-head attention implementation."""
    print("🧪 Unit Test: Multi-Head Attention...")

    # Test initialization
    embed_dim, num_heads = 64, 8
    mha = MultiHeadAttention(embed_dim, num_heads)

    # Check configuration
    assert mha.embed_dim == embed_dim
    assert mha.num_heads == num_heads
    assert mha.head_dim == embed_dim // num_heads

    # Test parameter counting (4 linear layers, each has weight + bias)
    params = mha.parameters()
    assert len(params) == 8, f"Expected 8 parameters (4 layers x 2), got {len(params)}"

    # Test forward pass
    batch_size, seq_len = 2, 6
    x = Tensor(np.random.randn(batch_size, seq_len, embed_dim))

    output = mha.forward(x)

    # Check output shape preservation
    assert output.shape == (batch_size, seq_len, embed_dim), f"Output shape {output.shape} incorrect"

    # Test with causal mask
    mask = Tensor(np.tril(np.ones((batch_size, seq_len, seq_len))))
    output_masked = mha.forward(x, mask)
    assert output_masked.shape == (batch_size, seq_len, embed_dim)

    # Test different head configurations
    mha_small = MultiHeadAttention(embed_dim=32, num_heads=4)
    x_small = Tensor(np.random.randn(1, 5, 32))
    output_small = mha_small.forward(x_small)
    assert output_small.shape == (1, 5, 32)

    print("✅ MultiHeadAttention works correctly!")

if __name__ == "__main__":
    test_unit_multihead_attention()

🧪 Unit Test: Multi-Head Attention...
✅ MultiHeadAttention works correctly!


In [15]:
def analyze_attention_complexity():
    """📊 Analyze attention computational complexity and memory scaling."""
    print("📊 Analyzing Attention Complexity...")

    # Test different sequence lengths to show O(n²) scaling
    embed_dim = 64
    sequence_lengths = [16, 32, 64, 128, 256]

    print("\nSequence Length vs Attention Matrix Size:")
    print("Seq Len | Attention Matrix | Memory (KB) | Complexity")
    print("-" * 55)

    for seq_len in sequence_lengths:
        # Calculate attention matrix size
        attention_matrix_size = seq_len * seq_len

        # Memory for attention weights (float32 = 4 bytes)
        attention_memory_kb = (attention_matrix_size * 4) / 1024

        # Total complexity (Q@K + softmax + weights@V)
        complexity = 2 * seq_len * seq_len * embed_dim + seq_len * seq_len

        print(f"{seq_len:7d} | {attention_matrix_size:14d} | {attention_memory_kb:10.2f} | {complexity:10.0f}")

    print(f"\n💡 KEY INSIGHT: Attention memory scales as O(n^2) with sequence length")
    print(f"🚀 For seq_len=1024, attention matrix alone needs {(1024*1024*4)/1024/1024:.1f} MB")

# Run the analysis
if __name__ == "__main__":
    analyze_attention_complexity()

📊 Analyzing Attention Complexity...

Sequence Length vs Attention Matrix Size:
Seq Len | Attention Matrix | Memory (KB) | Complexity
-------------------------------------------------------
     16 |            256 |       1.00 |      33024
     32 |           1024 |       4.00 |     132096
     64 |           4096 |      16.00 |     528384
    128 |          16384 |      64.00 |    2113536
    256 |          65536 |     256.00 |    8454144

💡 KEY INSIGHT: Attention memory scales as O(n^2) with sequence length
🚀 For seq_len=1024, attention matrix alone needs 4.0 MB


In [16]:
def analyze_attention_timing():
    """📊 Measure attention computation time vs sequence length."""
    print("\n📊 Analyzing Attention Timing...")

    embed_dim, num_heads = 64, 8
    sequence_lengths = [32, 64, 128, 256]

    print("\nSequence Length vs Computation Time:")
    print("Seq Len | Time (ms) | Ops/sec | Scaling")
    print("-" * 40)

    prev_time = None
    for seq_len in sequence_lengths:
        # Create test input
        x = Tensor(np.random.randn(1, seq_len, embed_dim))
        mha = MultiHeadAttention(embed_dim, num_heads)

        # Time multiple runs for stability
        times = []
        for _ in range(5):
            start_time = time.time()
            _ = mha.forward(x)
            end_time = time.time()
            times.append((end_time - start_time) * 1000)  # Convert to ms

        avg_time = np.mean(times)
        ops_per_sec = 1000 / avg_time if avg_time > 0 else 0

        # Calculate scaling factor vs previous
        scaling = avg_time / prev_time if prev_time else 1.0

        print(f"{seq_len:7d} | {avg_time:8.2f} | {ops_per_sec:7.0f} | {scaling:6.2f}x")
        prev_time = avg_time

    print(f"\n💡 KEY INSIGHT: Attention time scales roughly as O(n^2) with sequence length")
    print(f"🚀 This is why attention efficiency techniques are an active area of research")

# Run the analysis
if __name__ == "__main__":
    analyze_attention_timing()


📊 Analyzing Attention Timing...

Sequence Length vs Computation Time:
Seq Len | Time (ms) | Ops/sec | Scaling
----------------------------------------
     32 |     0.44 |    2285 |   1.00x
     64 |     0.38 |    2624 |   0.87x
    128 |     0.75 |    1332 |   1.97x
    256 |     4.22 |     237 |   5.62x

💡 KEY INSIGHT: Attention time scales roughly as O(n^2) with sequence length
🚀 This is why attention efficiency techniques are an active area of research


In [17]:
def analyze_attention_memory_overhead():
    """📊 Analyze memory overhead during training (forward + backward passes)."""
    print("\n📊 Analyzing Attention Memory Overhead During Training...")

    embed_dim, num_heads = 128, 8
    sequence_lengths = [128, 256, 512, 1024]

    print("\nMemory Overhead Analysis (Training vs Inference):")
    print("Seq Len | Forward | + Gradients | + Optimizer | Total Memory")
    print("-" * 65)

    for seq_len in sequence_lengths:
        # Forward pass memory (attention matrix)
        attention_matrix_mb = (seq_len * seq_len * 4) / (1024 * 1024)

        # Backward pass adds gradient storage (2× forward)
        backward_memory_mb = 2 * attention_matrix_mb

        # Optimizer state (Adam: +2× for momentum and velocity)
        optimizer_memory_mb = backward_memory_mb + 2 * attention_matrix_mb

        # Total = forward + gradients + optimizer state
        total_memory_mb = attention_matrix_mb + backward_memory_mb + optimizer_memory_mb

        print(f"{seq_len:7d} | {attention_matrix_mb:6.2f}MB | {backward_memory_mb:10.2f}MB | {optimizer_memory_mb:10.2f}MB | {total_memory_mb:11.2f}MB")

    print(f"\n💡 KEY INSIGHT: Training requires ~7x memory of inference (1x forward + 2x gradients + 4x optimizer state)")
    print(f"🚀 For GPT-3 (96 layers, 2048 context): ~6GB just for attention gradients!")

# Run the analysis
if __name__ == "__main__":
    analyze_attention_memory_overhead()


📊 Analyzing Attention Memory Overhead During Training...

Memory Overhead Analysis (Training vs Inference):
Seq Len | Forward | + Gradients | + Optimizer | Total Memory
-----------------------------------------------------------------
    128 |   0.06MB |       0.12MB |       0.25MB |        0.44MB
    256 |   0.25MB |       0.50MB |       1.00MB |        1.75MB
    512 |   1.00MB |       2.00MB |       4.00MB |        7.00MB
   1024 |   4.00MB |       8.00MB |      16.00MB |       28.00MB

💡 KEY INSIGHT: Training requires ~7x memory of inference (1x forward + 2x gradients + 4x optimizer state)
🚀 For GPT-3 (96 layers, 2048 context): ~6GB just for attention gradients!


In [18]:
def test_unit_attention_scenarios():
    """Test attention mechanisms in realistic scenarios."""
    print("🧪 Testing Attention Scenarios...")

    # Scenario 1: Small transformer block setup
    print("\n1. Small Transformer Setup:")
    embed_dim, num_heads, seq_len = 128, 8, 32

    # Create embeddings (simulating token embeddings + positional)
    embeddings = Tensor(np.random.randn(2, seq_len, embed_dim))

    # Multi-head attention
    mha = MultiHeadAttention(embed_dim, num_heads)
    attended = mha.forward(embeddings)

    print(f"   Input shape: {embeddings.shape}")
    print(f"   Output shape: {attended.shape}")
    print(f"   Parameters: {len(mha.parameters())} tensors")

    # Scenario 2: Causal language modeling
    print("\n2. Causal Language Modeling:")

    # Create causal mask (lower triangular)
    causal_mask = np.tril(np.ones((seq_len, seq_len)))
    mask = Tensor(np.broadcast_to(causal_mask, (2, seq_len, seq_len)))

    # Apply causal attention
    causal_output = mha.forward(embeddings, mask)

    print(f"   Masked output shape: {causal_output.shape}")
    print(f"   Causal mask applied: {mask.shape}")

    # Scenario 3: Compare attention patterns
    print("\n3. Attention Pattern Analysis:")

    # Create simple test sequence
    simple_embed = Tensor(np.random.randn(1, 4, 16))
    simple_mha = MultiHeadAttention(16, 4)

    # Get attention weights by calling the base function
    Q = simple_mha.q_proj.forward(simple_embed)
    K = simple_mha.k_proj.forward(simple_embed)
    V = simple_mha.v_proj.forward(simple_embed)

    # Reshape for single head analysis
    Q_head = Tensor(Q.data[:, :, :4])  # First head only
    K_head = Tensor(K.data[:, :, :4])
    V_head = Tensor(V.data[:, :, :4])

    _, weights = scaled_dot_product_attention(Q_head, K_head, V_head)

    print(f"   Attention weights shape: {weights.shape}")
    print(f"   Attention weights (first batch, 4x4 matrix):")
    weight_matrix = weights.data[0, :, :].round(3)

    # Format the attention matrix nicely
    print("     Pos→  0     1     2     3")
    for i in range(4):
        row_str = f"   {i}: " + " ".join(f"{weight_matrix[i,j]:5.3f}" for j in range(4))
        print(row_str)

    print(f"   Row sums: {weights.data[0].sum(axis=1).round(3)} (should be ~1.0)")

    # Scenario 4: Attention with masking visualization
    print("\n4. Causal Masking Effect:")

    # Apply causal mask to the simple example
    simple_mask = Tensor(np.tril(np.ones((1, 4, 4))))
    _, masked_weights = scaled_dot_product_attention(Q_head, K_head, V_head, simple_mask)

    print("   Causal attention matrix (lower triangular):")
    masked_matrix = masked_weights.data[0, :, :].round(3)
    print("     Pos→  0     1     2     3")
    for i in range(4):
        row_str = f"   {i}: " + " ".join(f"{masked_matrix[i,j]:5.3f}" for j in range(4))
        print(row_str)

    print("   Notice: Upper triangle is zero (can't attend to future)")

    print("\n✅ All attention scenarios work correctly!")

In [19]:
def test_module():
    """🧪 Module Test: Complete Integration

    Comprehensive test of entire attention module functionality.

    This final test runs before module summary to ensure:
    - All unit tests pass
    - Functions work together correctly
    - Module is ready for integration with TinyTorch
    """
    print("🧪 RUNNING MODULE INTEGRATION TEST")
    print("=" * 50)

    # Run all unit tests
    print("Running unit tests...")
    test_unit_attention_scores()
    test_unit_scale_scores()
    test_unit_apply_mask()
    test_unit_scaled_dot_product_attention()
    test_unit_split_heads()
    test_unit_merge_heads()
    test_unit_multihead_attention()

    print("\nRunning integration scenarios...")
    test_unit_attention_scenarios()

    print("\nRunning performance analysis...")
    analyze_attention_complexity()
    print("\nRunning memory overhead analysis...")
    analyze_attention_memory_overhead()

    print("\n" + "=" * 50)
    print("🎉 ALL TESTS PASSED! Module ready for export.")
    print("Run: tito module complete 12")

In [20]:
def demo_attention():
    """🎯 See attention compute relationships."""
    print("🎯 AHA MOMENT: Attention Finds Relationships")
    print("=" * 45)

    # Create Q, K, V for 4 tokens with 8-dim embeddings
    Q = Tensor(np.random.randn(1, 4, 8))
    K = Tensor(np.random.randn(1, 4, 8))
    V = Tensor(np.random.randn(1, 4, 8))

    # Compute attention
    output, weights = scaled_dot_product_attention(Q, K, V)

    print(f"Sequence length: 4 tokens")
    print(f"Embedding dim:   8")
    print(f"\nAttention weights shape: {weights.shape}")
    print(f"Each token attends to all 4 positions!")

    print(f"\nToken 0 attention: {weights.data[0, 0, :].round(2)}")
    print("(sums to 1.0 - it's a probability distribution)")

    print("\n✨ Attention lets tokens communicate!")

In [21]:
if __name__ == "__main__":
    test_module()
    print("\n")
    demo_attention()

🧪 RUNNING MODULE INTEGRATION TEST
Running unit tests...
🧪 Unit Test: Attention Scores...
Tensor([[[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]])
Tensor([[[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]])
Tensor([[[4. 4. 4.]
  [4. 4. 4.]
  [4. 4. 4.]]])
✅ Attention scores: correct shapes and values!
🧪 Unit Test: Score Scaling...
✅ Score scaling works correctly!
🧪 Unit Test: Causal Masking...
✅ Causal masking works correctly!
🧪 Unit Test: Scaled Dot-Product Attention...
✅ scaled_dot_product_attention works correctly!
🧪 Unit Test: Split Heads...
✅ Split heads: correct 4D shape!
🧪 Unit Test: Merge Heads...
✅ Merge heads: correct 3D shape and round-trip!
🧪 Unit Test: Multi-Head Attention...
✅ MultiHeadAttention works correctly!

Running integration scenarios...
🧪 Testing Attention Scenarios...

1. Small Transformer Setup:
   Input shape: (2, 32, 128)
   Output shape: (2, 32, 128)
   Parameters: 8 tensors

2. Causal Language Modeling:
   Masked output shape: (2, 32, 128)
   Causal mask appl